In [ ]:
import sys
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score

# 1. Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.models.cvae import CVAE

# 2. Setup Configuration (Must match the training config exactly)
cvae_config = {
    'input_shape': (1738, 1),
    'num_classes': 6,
    'latent_dim': 32,
    'loss_weights': {
        'beta': 2,  # Slightly > 1 forces better disentanglement of latent space
        'alpha': 0.2  # Classification loss weight 
    },
    'encoder': {
        'filters': [32, 64, 128],  # Increasing filter depth
        'kernels': [11, 5, 3],     # Decreasing receptive field (Broad to Fine)
        'strides': [2, 2, 2],      # Downsampling 1738 -> 869 -> 435 -> 218
        'use_global_pooling': False, # Option B: Flatten (preserves exact spatial peak positions)
        'use_skip_gap': False
    },
    'decoder': {
        'initial_filters': 128,
        'use_dense_projection': True, # Projects Latent Space into spatial features
        'filters': [64, 32, 16],      # Upsampling symmetrically
        'kernels': [3, 5, 11],
        'strides': [2, 2, 2]
    }
}

# 3. Initialize Data Loader
DATA_PATH = "../data/processed/glioma_dataset.h5"
SPLIT_FILE = "../data/splits/manual_splits.json"
print("Initializing Loader...")
loader = GliomaDataLoader(data_path=DATA_PATH, split_file=SPLIT_FILE, memory_mapped=False, tumor_only=True)

# 4. Setup Output / Input Directories
MODEL_DIR = "../results/models/cvae"
cbr_results = {
    'Classification_by_Recon': []
}

print("\n🚀 Starting Generative Inference: Classification by Reconstruction...")

# 5. Main Inference Loop
for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n{'='*40}")
    print(f"🔄 Evaluating {fold_name}...")
    
    # Unpack Real Validation Data
    X_val = fold_data['val'][0].astype(np.float32)
    y_val = fold_data['val'][1]
    
    # Fix 1-indexed to 0-indexed if necessary
    if np.min(y_val) == 1:
        y_val = y_val - 1
        
    X_val_exp = np.expand_dims(X_val, axis=-1) # Shape: (N, 1738, 1)
    n_samples = X_val.shape[0]
    
    # --- MODEL INITIALIZATION TRICK (From yesterday's Dry Run) ---
    tf.keras.backend.clear_session()
    cvae_dummy = CVAE(cvae_config)
    cvae_dummy.compile(optimizer='adam')
    
    # Pass a tiny dummy batch to perfectly initialize all tensor shapes in memory
    dummy_x = tf.zeros((2, 1738, 1))
    dummy_y = tf.convert_to_tensor(tf.keras.utils.to_categorical([0, 1], num_classes=6), dtype=tf.float32)
    cvae_dummy.train_step((dummy_x, dummy_y))
    
    # Load the specific weights for this fold
    cvae_dummy.encoder.load_weights(f"{MODEL_DIR}/encoder_{fold_name}.weights.h5")
    cvae_dummy.decoder.load_weights(f"{MODEL_DIR}/decoder_{fold_name}.weights.h5")
    
    encoder = cvae_dummy.encoder
    decoder = cvae_dummy.decoder
    
    # --- PHASE 1: ENCODE ---
    print("  > Extracting latent physics (z)...")
    z_mean, _, _, _ = encoder.predict(X_val_exp, batch_size=256, verbose=0)
    
    # --- PHASE 2: DECODE (6 Times) ---
    print("  > Testing 6-class reconstruction MSE...")
    mse_scores = np.zeros((n_samples, 6), dtype=np.float32)
    
    for class_idx in range(6):
        requested_classes = tf.convert_to_tensor(
            tf.keras.utils.to_categorical([class_idx] * n_samples, num_classes=6),
            dtype=tf.float32
        )
        
        # "If this spectrum was Class X, what should it look like?"
        reconstructed_spectra = decoder.predict([z_mean, requested_classes], batch_size=256, verbose=0)
        reconstructed_spectra = np.squeeze(reconstructed_spectra, axis=-1)
        
        # Calculate Mean Squared Error across all 1738 bands
        mse = np.mean((X_val - reconstructed_spectra)**2, axis=1)
        mse_scores[:, class_idx] = mse
        
    # --- PHASE 3: CLASSIFY ---
    # The winner is the class with the lowest MSE
    y_pred_cbr = np.argmin(mse_scores, axis=1)
    
    acc = accuracy_score(y_val, y_pred_cbr)
    cbr_results['Classification_by_Recon'].append(acc)
    print(f"  ✅ {fold_name} CbR Accuracy: {acc:.4f}")

# --- Print Final Summary ---
print("\n🎉 === CLASSIFICATION BY RECONSTRUCTION COMPLETE ===")
mean_acc = np.mean(cbr_results['Classification_by_Recon'])
std_acc = np.std(cbr_results['Classification_by_Recon'])
print(f"Classification by Recon: {mean_acc:.4f} ± {std_acc:.4f}")

Initializing Loader...
📂 Loading data from ..\data\processed\glioma_dataset.h5...
🔬 'tumor_only=True': Filtering out healthy tissue...
   ✅ Filtered to 88487 Pure Tumor Spectra.

🚀 Starting Generative Inference: Classification by Reconstruction...

🔄 Evaluating fold_0...


KeyError: 'loss_weights'